# Robinhood performance (tracker export)

Parameterized notebook for **Reports → Finance → Robinhood**. Data comes from CSV import in tracker-pg (no Robinhood API).

Run locally with JupyterLab or via `robinhood-notebook-svc` (`POST /v1/render`).

In [ ]:
# Papermill parameters (overridden by robinhood-notebook-svc or manual run)
bundle_path = ""
year = 0

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

from lib.tracker_robinhood import load_bundle, monthly_pnl_frame, performance_summary, transactions_frame

if not bundle_path:
    raise ValueError("Set bundle_path to a notebook-bundle JSON file (download from Reports → Robinhood).")

bundle = load_bundle(bundle_path)
display_year = year or bundle.get("year")
print(f"Year: {display_year}  rows: {bundle.get('transactionRowCount')}  truncated: {bundle.get('transactionsTruncated')}")

In [ ]:
summary = performance_summary(bundle)
total = summary.get("totalRealizedPnL")
win_rate = summary.get("winRate")
print(f"Total realized P&L: ${float(total or 0):,.2f}")
print(f"Win rate: {float(win_rate or 0) * 100:.1f}%")

In [ ]:
import matplotlib.pyplot as plt

monthly = monthly_pnl_frame(bundle)
if monthly.empty:
    print("No monthly P&L in bundle.")
else:
    labels = monthly.get("monthLabel", monthly.index.astype(str))
    pnl = monthly["realizedPnL"].astype(float)
    colors = ["#2e7d32" if v >= 0 else "#c62828" for v in pnl]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(labels, pnl, color=colors)
    ax.axhline(0, color="#666", linewidth=0.8)
    ax.set_title(f"Monthly realized P&L ({display_year})")
    ax.set_ylabel("USD")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
tx = transactions_frame(bundle)
if tx.empty:
    print("No transactions in bundle.")
else:
    display(tx.head(20))